# Prompt engineering to Agentic RAG 🤖🧠
### LangChain RAG Agent — Prompt Engineering with a Hugging Face Dataset
---

> This notebook builds a Retrieval-Augmented Generation agent with LangChain and the public AG News dataset from Hugging Face.

> The goal is to keep the same retrieval pipeline while changing the system prompt to compare grounding, citations, format, length, tone, audience adaptation, clarification behavior, and
> insufficient-context handling.
> AG News is a historical benchmark dataset. Treat it as a news archive, not a live news service.

## Learning objectives

1. Load a public Hugging Face dataset.
2. Convert rows into LangChain documents.
3. Build an in-memory vector store.
4. Expose retrieval as an agent tool.
5. Compare weak and optimized prompts.
6. Test ambiguity and insufficient-context handling.
7. Evaluate output quality.
8. Demonstrate and apply LangChain implementations to improve retrieval quality with [query transformation](https://www.langchain.com/blog/query-transformations).
9. Apply query expansion, query rewriting, and Hypothetical Document Embeddings (HyDE) to improve recall and retrieve more relevant documents.
10. Run the agent using **all** query transformation techniques combined (baseline + rewrite + expansion + HyDE).


**Install dependencies**

In [ ]:
%pip install -U datasets langchain langchain-openai langchain-community langchain-text-splitters python-dotenv colorama


**Environment setup**

In [ ]:
import os
from getpass import getpass
from colorama import Fore, Style
from dotenv import load_dotenv

load_dotenv(override=True)

if not os.getenv('OPENAI_API_KEY'):
    os.environ['OPENAI_API_KEY'] = getpass('Enter your OpenAI API key: ')
else:
    print(f'{Fore.GREEN}OpenAI API key loaded.{Style.RESET_ALL}')


**Initialize the model**

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model("openai:gpt-4.1-mini", temperature=0, timeout=30, max_retries=3)

print(f"{Fore.CYAN} Model initialized.{Style.RESET_ALL}")


**Defining Prompts**

In [ ]:
WEAK_PROMPT="""You are a news assistant. Use the search tool and answer the user's question."""

GROUNDED_PROMPT="""You are a grounded news-archive assistant. The knowledge base is historical, not real-time.
Use search_optimized_news_archive with strategy='auto'. Use only retrieved evidence, cite [Source N], and never present archive data as current.
If evidence is insufficient, say so. Begin directly, add up to three cited bullets, finish with one limitation, and use no more than 180 words."""

BEGINNER_FRIENDLY_PROMPT="""You are a teacher explaining historical news to beginners. Use search_optimized_news_archive with strategy='auto'.
Use only retrieved evidence. Explain difficult terms, use at most three bullets, include one example, cite [Source N], end with 'In simple terms, ...',
and write 140 to 180 words."""

EXECUTIVE_PROMPT="""You are preparing a briefing for a business leader.
Use search_optimized_news_archive with strategy='extractive'. Return exactly: ## Headline, ## Key Signals, ## Business Relevance, ## Data Limitation.
Use only retrieved evidence and no more than 140 words."""

CLARIFYING_PROMPT="""You are a grounded news-archive assistant. If the request is ambiguous, ask one concise clarification question and do not search.
Otherwise use search_optimized_news_archive with strategy='auto', cite [Source N], and state when evidence is insufficient."""

ANALYTICAL_PROMPT="""You are an analytical research assistant. Use search_optimized_news_archive with strategy='summarize'. Return exactly:
## Main Finding, ## Evidence, ## Interpretation, ## Limitation. Use only retrieved evidence, neutral language, and no more than 220 words."""


**Load the AG News dataset**

Using public dataset from Hugging Face: [AG News](https://huggingface.co/datasets/fancyzhx/ag_news)
A small slice keeps the demo fast and inexpensive.

In [ ]:
from datasets import load_dataset

dataset = load_dataset('fancyzhx/ag_news', split='train[:500]')
print(dataset)
print(dataset[0])


**Convert data into LangChain documents**

In [ ]:
from langchain_core.documents import Document

LABELS = {0: 'World', 1: 'Sports', 2: 'Business', 3: 'Science and Technology'}

documents = []
for index, row in enumerate(dataset):
    label = int(row['label'])
    documents.append(
        Document(
            page_content=row['text'].strip(),
            metadata={
                'source': f'ag_news_{index}',
                'category': LABELS[label],
                'dataset': 'AG News',
                'record_id': index,
            },
        )
    )

print(f'Documents created: {len(documents)}')
print(documents[0])


**Split, embed, and index the documents**

In [ ]:
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents)

embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = InMemoryVectorStore(embedding=embeddings)
vector_store.add_documents(chunks)
retriever = vector_store.as_retriever(search_kwargs={'k': 5})

print(f'Documents: {len(documents)}')
print(f'Chunks: {len(chunks)}')


**Format retrieved documents**

In [ ]:
def format_documents(docs: list[Document]) -> str:
    if not docs:
        return 'No relevant articles were retrieved.'

    formatted = []
    for position, doc in enumerate(docs, start=1):
        formatted.append(
            f'[Source {position}]\n'
            f"Record: {doc.metadata.get('source', 'unknown')}\n"
            f"Category: {doc.metadata.get('category', 'unknown')}\n"
            f'Article: {doc.page_content}'
        )
    return '\n\n'.join(formatted)


**Helper and utility functions**

In [ ]:
import tiktoken
TOKENIZER=tiktoken.encoding_for_model("gpt-4.1-mini")

def count_tokens(text:str)->int:
    """Count the number of tokens in the given text using the specified tokenizer."""
    return len(TOKENIZER.encode(text))

def document_tokens(doc:Document)->int:
    """Count the number of tokens in the given Document's page content."""
    return count_tokens(doc.page_content)

def inspect_token_usage(docs:list[Document], label="Context"):
    """Inspect token usage for a list of documents and print statistics."""
    counts=[document_tokens(d) for d in docs]
    stats={
        "label":label, "documents":len(docs), "total_tokens":sum(counts),
        "average_tokens":round(sum(counts)/len(counts),1) if counts else 0,
        "largest_document":max(counts,default=0),
    }
    print(stats)
    return stats

def format_documents(docs:list[Document])->str:
    """Format a list of Document objects into a readable string representation."""
    if not docs:
        return "No relevant articles were retrieved."
    parts=[]
    for i,doc in enumerate(docs,1):
        parts.append(
            f"[Source {i}]\nRecord: {doc.metadata.get('source','unknown')}\n"
            f"Category: {doc.metadata.get('category','unknown')}\nArticle: {doc.page_content}"
        )
    return "\n\n".join(parts)


**Pretty-print helper for agent results**

In [ ]:
import textwrap
from colorama import Fore, Style

def print_agent_result(name: str, result, width: int = 100):
    """Pretty-print an (answer, metrics) tuple returned by ask_agent."""
    answer, metrics = result

    print("\n" + "=" * width)
    print(f"{Fore.CYAN}{Style.BRIGHT}{name.upper()}{Style.RESET_ALL}")
    print("=" * width)

    # Metrics line
    action = metrics.get("action", "n/a")
    before = metrics.get("before_tokens", "n/a")
    after = metrics.get("after_tokens", "n/a")
    reduction = metrics.get("token_reduction", "n/a")

    # Answer, shortened and word-wrapped for readability
    print(f"\n{Fore.GREEN}[Answer]{Style.RESET_ALL}")
    clean_answer = answer.replace("\\n", "\n")
    max_chars = 400
    if len(clean_answer) > max_chars:
        clean_answer = clean_answer[:max_chars].rsplit(" ", 1)[0] + " […]"
    wrapped = textwrap.fill(clean_answer, width=width, replace_whitespace=False)
    print(wrapped)

    print(f"{Fore.YELLOW}[Context]{Style.RESET_ALL} "
          f"action={action}  tokens_before={before}  tokens_after={after}  reduction={reduction}")


**Context Optimization**

In [ ]:
import re
from math import sqrt
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

SUMMARY_PROMPT=ChatPromptTemplate.from_messages([
    ("system",
     "Compress retrieved news evidence for a downstream RAG answer. "
     "Preserve facts, source labels exactly as [Source N], separate events, and limitations. "
     "Do not add outside information. Return no more than {target_words} words."),
    ("human","Question:\n{question}\n\nRetrieved evidence:\n{context}\n\nCompressed evidence:"),
])
summary_chain=SUMMARY_PROMPT | model | StrOutputParser()

#*** ----------------------------------------- ***
#*** TRUNCATION of documents by token count
#*** Truncation documents based on token count.
#*** Keeps the highest-ranked documents that fit a fixed token budget. It is fast, deterministic, and adds no LLM call. Its weakness is that useful lower-ranked evidence may be excluded.
#*** ----------------------------------------- ***

def truncate_documents_by_tokens(docs:list[Document], max_tokens=900):
    """Truncate a list of Document objects to fit within a specified token limit."""
    selected=[]; excluded=[]; used=0
    for doc in docs:
        n=document_tokens(doc)
        if used+n <= max_tokens:
            selected.append(doc); used+=n
        else:
            excluded.append(doc)
    return selected,{
        "budget":max_tokens,"used_tokens":used,
        "selected_documents":len(selected),"excluded_documents":len(excluded),
    }

query="technology companies and business risks"
raw_docs=retriever.invoke(query)
truncated_docs,trunc_report=truncate_documents_by_tokens(raw_docs,900)
inspect_token_usage(raw_docs,"Before truncation")
inspect_token_usage(truncated_docs,"After truncation")
print(trunc_report)


#*** ----------------------------------------- ***
#*** COMPRESSION of documents by extractive summarization
#*** Extractive compression keeps query-relevant sentences while preserving original wording. It reduces context without a generation call,
#*** although sentence-level similarity can remove useful background.
#*** ----------------------------------------- ***

def cosine_similarity(a,b):
    numerator=sum(x*y for x,y in zip(a,b))
    na=sqrt(sum(x*x for x in a)); nb=sqrt(sum(y*y for y in b))
    return numerator/(na*nb) if na and nb else 0.0

def split_sentences(text):
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+",text) if s.strip()]

def compress_documents(query,docs,threshold=0.20,max_sentences_per_doc=2):
    """Compress documents by extracting the most relevant sentences based on cosine similarity to the query.
    Sentences across all documents are embedded in a single batched call to embed_documents,
    rather than one embedding call per document, to keep latency down as retrieval breadth grows."""
    qv=embeddings.embed_query(query)

    doc_sentences=[split_sentences(doc.page_content) for doc in docs]
    all_sentences=[s for sentences in doc_sentences for s in sentences]

    if not all_sentences:
        return []

    all_vectors=embeddings.embed_documents(all_sentences)

    output=[]
    idx=0
    for doc, sentences in zip(docs, doc_sentences):
        n=len(sentences)
        vectors=all_vectors[idx: idx+n]
        idx+=n
        if not sentences:
            continue
        ranked=sorted([(cosine_similarity(qv,v),s) for s,v in zip(sentences,vectors)], reverse=True)
        kept=[s for score,s in ranked[:max_sentences_per_doc] if score>=threshold]
        if kept:
            output.append(Document(page_content=" ".join(kept), metadata={**doc.metadata,"compression":"extractive"}))
    return output

compress_docs =compress_documents(query,raw_docs)
compress_docs,_ =truncate_documents_by_tokens(compress_docs,900)
inspect_token_usage(raw_docs,"Raw retrieval")
inspect_token_usage(compress_docs,"Documents compression")
print(format_documents(compress_docs[:3]))

#*** ----------------------------------------- ***
#***  SUMMARIZATION
#***  Summarization can synthesize multiple articles and sharply reduce context, but it adds latency and cost and may blur source boundaries.
#***  Apply it conditionally and require the model to preserve source labels and uncertainty.
#*** ----------------------------------------- ***

def summarize_context(question,docs,target_words=180):
    return summary_chain.invoke({
        "question":question,"context":format_documents(docs),"target_words":target_words
    })

def summarize_if_needed(question,docs,threshold_tokens=1000,target_words=180):
    raw=format_documents(docs); before=count_tokens(raw)
    if before<=threshold_tokens:
        return raw,{"strategy":"full_context","before_tokens":before,"after_tokens":before}
    summary=summarize_context(question,docs,target_words)
    return summary,{"strategy":"llm_summary","before_tokens":before,"after_tokens":count_tokens(summary)}

summarized_context,summary_report=summarize_if_needed(query,raw_docs,700,180)
print(summary_report)
print(summarized_context)


**LangChain contextual compression (optional)**

`ContextualCompressionRetriever` wraps a base retriever and applies a compressor. Here a local cross-encoder reranks twelve candidates down to five. It often improves precision, but adds inference time and downloads a model on first use.

In [ ]:
try:
    from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
    from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
    from langchain_community.cross_encoders import HuggingFaceCrossEncoder

    cross_encoder=HuggingFaceCrossEncoder(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
    reranker=CrossEncoderReranker(model=cross_encoder,top_n=5)
    compression_retriever=ContextualCompressionRetriever(
        base_retriever=retriever,base_compressor=reranker
    )
    langchain_compressed_docs=compression_retriever.invoke(query)
    inspect_token_usage(langchain_compressed_docs,"LangChain contextual compression")
except Exception as exc:
    langchain_compressed_docs=[]
    print("Optional demo skipped:",exc)


**Context Optimization**

In [ ]:
def optimize_context(
    query,
    docs,
    max_context_tokens=900,
    summarize_threshold=700,
):
    """
    Optimize retrieved context automatically based on token count.

    Flow:
    1. Keep the full context if it already fits.
    2. Apply extractive compression when it is too large.
    3. Apply token-aware truncation.
    4. Summarize only if the context is still above the summarization threshold.
    """

    before_context = format_documents(docs)
    before_tokens = count_tokens(before_context)

    details = {
        "action": "full_context",
        "documents_before": len(docs),
    }

    # 1. No optimization needed
    if before_tokens <= max_context_tokens:
        context = before_context

    else:
        # 2. Remove less relevant sentences
        compressed_docs = compress_documents(
            query,
            docs,
        )

        details["action"] = "extractive_compression"
        details["documents_after_compression"] = len(compressed_docs)

        # Fall back to original docs if compression removes everything
        if not compressed_docs:
            compressed_docs = docs
            details["compression_fallback"] = True

        # 3. Enforce the context token budget
        selected_docs, truncation_report = truncate_documents_by_tokens(
            compressed_docs,
            max_tokens=max_context_tokens,
        )

        context = format_documents(selected_docs)

        details.update(truncation_report)
        details["action"] = "compress_then_truncate"

        # 4. Summarize only if the formatted context is still too large
        if count_tokens(context) > summarize_threshold:
            context = summarize_context(
                query,
                selected_docs,
                target_words=180,
            )

            details["action"] = "compress_truncate_then_summarize"

    after_tokens = count_tokens(context)

    return context, {
        "before_tokens": before_tokens,
        "after_tokens": after_tokens,
        "token_reduction": before_tokens - after_tokens,
        **details,
    }


## Query Transformation

The retrieval pipeline above always embeds the user's question exactly as written. But the raw question is
often not the best possible search query: it may be vague, conversational, or phrased differently from the
vocabulary used in the archive itself. [Query transformation](https://www.langchain.com/blog/query-transformations)
techniques rewrite or expand the query *before* retrieval to improve recall — the goal is to retrieve more of
the relevant documents that a naive similarity search on the raw question would miss.

Three techniques are added below, each as a self-contained module that reuses the existing `model`,
`embeddings`, `vector_store`, and `retriever` objects already defined — nothing above this point is modified:

1. **Query Rewriting** — reformulate the question into a clearer, retrieval-optimized query.
2. **Query Expansion** — generate several alternate phrasings and merge retrieval results across all of them.
3. **HyDE (Hypothetical Document Embeddings)** — generate a hypothetical answer passage and embed *that*
   instead of the question, since an answer-shaped embedding often sits closer to real answer documents.

### Query Rewriting

This module asks the model to turn the user's question into a single, clear, keyword-rich query, better
suited for similarity search than a casual or ambiguous phrasing. It's useful when the raw question is vague
(e.g. "what's going on with tech stocks lately") and the archive itself uses more specific vocabulary.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

REWRITE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Rewrite the user's question into a single, clear, keyword-rich search query "
     "optimized for retrieval over a historical news archive. "
     "Do not answer the question. Do not add explanations. "
     "Return only the rewritten query, nothing else."),
    ("human", "{question}"),
])
rewrite_chain = REWRITE_PROMPT | model | StrOutputParser()

def rewrite_query(question: str) -> str:
    """Rewrite a raw user question into a retrieval-optimized query."""
    return rewrite_chain.invoke({"question": question}).strip()

# demo
example_question = "what's going on with tech stocks lately"
rewritten = rewrite_query(example_question)
print(f"Original:  {example_question}")
print(f"{Fore.MAGENTA}Rewritten: {rewritten}{Fore.RESET}")


### Query Expansion

This module generates several alternate phrasings of the same question, retrieves documents for each variant,
and merges the results while de-duplicating by source. This widens recall by covering different terminology
the corpus might use for the same underlying concept, at the cost of extra retrieval calls.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

EXPANSION_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Generate {n} different search queries that explore different phrasings, "
     "synonyms, or angles of the user's question, for retrieval over a historical "
     "news archive. Return exactly {n} queries, one per line, no numbering, no extra text."),
    ("human", "{question}"),
])
expansion_chain = EXPANSION_PROMPT | model | StrOutputParser()

def expand_query(question: str, n: int = 3) -> list[str]:
    """Generate n alternate phrasings of the question."""
    raw = expansion_chain.invoke({"question": question, "n": n})
    queries = [line.strip("-\u2022 ").strip() for line in raw.split("\n") if line.strip()]
    return queries[:n]

def retrieve_with_expansion(question: str, n: int = 3, k: int = 5) -> list[Document]:
    """Retrieve using the original question plus n expanded variants, then dedupe."""
    queries = [question] + expand_query(question, n=n)
    seen_sources = set()
    merged_docs = []
    for q in queries:
        for doc in retriever.invoke(q):
            source = doc.metadata.get("source")
            if source not in seen_sources:
                seen_sources.add(source)
                merged_docs.append(doc)
    return merged_docs[:k]

# demo
example_question = "technology companies and business risks"
expanded_queries = expand_query(example_question)
print(f"{Fore.MAGENTA}Expanded queries: {expanded_queries}{Fore.RESET}")
expanded_docs = retrieve_with_expansion(example_question)
inspect_token_usage(expanded_docs, "Query expansion retrieval")


### HyDE (Hypothetical Document Embeddings)

This module generates a short hypothetical answer passage for the question, then embeds *that generated
passage* instead of the raw question and retrieves against it. An answer-shaped embedding often sits closer
in vector space to the real answer documents than a question-shaped embedding does, which can improve recall
for questions phrased very differently from how the archive itself describes the same event.

In [ ]:
HYDE_PROMPT = ChatPromptTemplate.from_messages([
    ("system",
     "Write a short, factual-sounding hypothetical news article passage that "
     "would answer the user's question, as if it were an excerpt from a "
     "historical news archive. Do not mention that it is hypothetical. "
     "Keep it to 2-3 sentences."),
    ("human", "{question}"),
])
hyde_chain = HYDE_PROMPT | model | StrOutputParser()

def hyde_retrieve(question: str, k: int = 5) -> list[Document]:
    """Generate a hypothetical answer passage and retrieve using its embedding."""
    hypothetical_doc = hyde_chain.invoke({"question": question})
    print(hypothetical_doc)
    hyde_vector = embeddings.embed_query(hypothetical_doc)
    print(hyde_vector)
    return vector_store.similarity_search_by_vector(hyde_vector, k=k)

# demo
example_question = "technology companies and business risks"
hyde_docs_demo = hyde_retrieve(example_question)
inspect_token_usage(hyde_docs_demo, "HyDE retrieval")


### Unified dispatcher — apply every technique automatically

Instead of choosing one technique per call, `retrieve_with_all_transformations` always runs the baseline
retrieval, rewriting, expansion, and HyDE together in a single pass, then merges the combined results and
de-duplicates by source. There's no `technique` parameter and no conditional branching to select a path —
every technique fires on every call, and the dispatcher's only job is to merge and de-duplicate what comes
back. The new agent tool, `search_news_archive_transformed`, exposes this combined retrieval with a single
`query` argument.

In [ ]:
from langchain.tools import tool

def merge_unique_documents(*document_lists: list[Document]) -> list[Document]:
    """Merge several lists of Documents into one, de-duplicated by source, preserving first-seen order."""
    seen_sources = set()
    merged = []
    for docs in document_lists:
        for doc in docs:
            source = doc.metadata.get("source")
            is_new = source not in seen_sources
            seen_sources.add(source)
            merged.extend([doc] * is_new)
    return merged


def retrieve_with_all_transformations(question: str, k: int = 5) -> tuple[list[Document], dict]:
    """
    Retrieve documents by automatically applying every query transformation technique
    (baseline, rewriting, expansion, HyDE) and merging the combined, de-duplicated results.
    """
    rewritten_query = rewrite_query(question)
    expanded_queries = expand_query(question)

    baseline_docs = retriever.invoke(question)
    rewritten_docs = retriever.invoke(rewritten_query)
    expanded_docs = retrieve_with_expansion(question, k=k)
    hyde_docs = hyde_retrieve(question, k=k)

    merged_docs = merge_unique_documents(baseline_docs, rewritten_docs, expanded_docs, hyde_docs)[:k]

    details = {
        "rewritten_query": rewritten_query,
        "expanded_queries": expanded_queries,
        "baseline_documents": len(baseline_docs),
        "rewrite_documents": len(rewritten_docs),
        "expansion_documents": len(expanded_docs),
        "hyde_documents": len(hyde_docs),
        "documents_retrieved": len(merged_docs),
        "sources": [d.metadata.get("source") for d in merged_docs],
    }
    return merged_docs, details


### Compare recall/overlap across techniques

A quick evaluation cell: for a given question, retrieve with each technique's own function individually
(baseline, rewrite, expansion, HyDE) purely to measure their *individual* contribution, then compare each
against the combined result from `retrieve_with_all_transformations`. This shows how many unique documents
each technique surfaces versus the baseline, and how many additional documents the combined automatic pass
picks up in total — a concrete, numeric look at the recall improvement from applying every technique
together rather than any single one.

In [ ]:
import pandas as pd

def compare_retrieval_techniques(question: str, k: int = 5) -> pd.DataFrame:
    """Compare each individual technique's retrieval against the combined, automatic result."""
    baseline_docs = retriever.invoke(question)
    rewritten_docs = retriever.invoke(rewrite_query(question))
    expanded_docs = retrieve_with_expansion(question, k=k)
    hyde_docs = hyde_retrieve(question, k=k)
    combined_docs, _ = retrieve_with_all_transformations(question, k=k)

    baseline_sources = {d.metadata.get("source") for d in baseline_docs}

    individual_results = {
        "baseline": baseline_docs,
        "rewrite": rewritten_docs,
        "expansion": expanded_docs,
        "hyde": hyde_docs,
        "combined_auto": combined_docs,
    }

    rows = []
    for name, docs in individual_results.items():
        sources = {d.metadata.get("source") for d in docs}
        rows.append({
            "technique": name,
            "documents_retrieved": len(docs),
            "unique_sources": len(sources),
            "new_vs_baseline": len(sources - baseline_sources),
        })
    return pd.DataFrame(rows)

comparison_question = "technology companies and business risks"
compare_retrieval_techniques(comparison_question)


**Create the retrieval tool**

In [ ]:
from langchain.tools import tool

@tool
def search_news_archive(query: str) -> str:
    """Search the historical AG News archive for relevant articles."""
    retrieved_documents = retriever.invoke(query)
    if not retrieved_documents:
        return 'STATUS: NO_RESULTS\nNo relevant news articles were found.'
    return 'STATUS: RESULTS_FOUND\n\n' + format_documents(retrieved_documents)


@tool
def search_optimized_news_archive(query:str,strategy:str="auto")->str:
    """Search AG News and optimize context. Strategies: full, truncate, extractive, summarize, auto."""
    docs=retriever.invoke(query)
    if not docs: return "STATUS: NO_RESULTS\nNo relevant articles were found."
    if strategy not in {"full","truncate","extractive","summarize","auto"}: strategy="auto"
    context,report=optimize_context(query,docs,strategy)
    return (f"STATUS: RESULTS_FOUND\nCONTEXT_STRATEGY: {report['strategy']}\n"
            f"TOKENS_BEFORE: {report['before_tokens']}\nTOKENS_AFTER: {report['after_tokens']}\n"
            f"TOKEN_REDUCTION: {report['token_reduction']}\n\n{context}")


@tool
def search_news_archive_transformed(query: str) -> str:
    """Search AG News using every query transformation technique (baseline, rewrite, expansion, HyDE) merged together."""
    docs, details = retrieve_with_all_transformations(query)
    if not docs:
        return "STATUS: NO_RESULTS\nNo relevant articles were found."
    context = format_documents(docs)
    return (
        "STATUS: RESULTS_FOUND\n"
        f"REWRITTEN_QUERY: {details['rewritten_query']}\n"
        f"EXPANDED_QUERIES: {details['expanded_queries']}\n"
        f"DOCUMENTS_RETRIEVED: {details['documents_retrieved']}\n\n"
        f"{context}"
    )


**Create the agent**

In [ ]:
from langchain.agents import create_agent

# switch prompt when you want to change the agent's behavior

rag_agent = create_agent(
    model=model,
    tools=[search_news_archive],
    system_prompt=GROUNDED_PROMPT,
)

def ask_agent(agent, question: str):
    """
    Retrieve documents, optimize the context,
    then invoke the LangChain agent.
    """

    # Retrieve documents
    docs = retriever.invoke(question)

    # Optimize retrieved context
    context, metrics = optimize_context(
        query=question,
        docs=docs,
    )

    # Invoke the agent
    result = agent.invoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": f"""
            Retrieved context:

            {context}

            Question:
            {question}
            """
                            }
                        ]
                    }
                )

    answer = result["messages"][-1].content

    return answer, metrics

# replace with

agents = {
    "Weak": create_agent(model=model, tools=[search_news_archive], system_prompt=WEAK_PROMPT),
    "Grounded": create_agent(model=model, tools=[search_news_archive], system_prompt=GROUNDED_PROMPT),
    "Beginner": create_agent(model=model, tools=[search_news_archive], system_prompt=BEGINNER_FRIENDLY_PROMPT),
    "Executive": create_agent(model=model, tools=[search_news_archive], system_prompt=EXECUTIVE_PROMPT),
    "Clarifying": create_agent(model=model, tools=[search_news_archive], system_prompt=CLARIFYING_PROMPT),
    "Analytical": create_agent(model=model, tools=[search_news_archive], system_prompt=ANALYTICAL_PROMPT),
}


## Run agent with all query transformations combined

This section adds a sixth pipeline alongside the five prompt-variant agents above: instead of a single-query
retrieval, it retrieves using baseline + rewrite + expansion + HyDE merged together
(`retrieve_with_all_transformations`), then runs the same context-optimization and agent-invocation flow as
`ask_agent`. Nothing above this point is modified — `ask_agent`, `agents`, and both original tools are
untouched; this only adds a new tool, a new agent entry, and a new runner function.

In [ ]:
agents["Transformed"] = create_agent(
    model=model,
    tools=[search_news_archive_transformed],
    system_prompt=GROUNDED_PROMPT,
)

def ask_agent_transformed(agent, question: str):
    """
    Same flow as ask_agent, but retrieves using every query transformation
    technique (baseline, rewrite, expansion, HyDE) merged together, instead
    of a single-query retrieval.
    """
    docs, transform_details = retrieve_with_all_transformations(question)
    context, metrics = optimize_context(query=question, docs=docs)
    metrics.update(transform_details)

    result = agent.invoke({
        "messages": [
            {
                "role": "user",
                "content": f"""
        Retrieved context:

        {context}

        Question:
        {question}
        """
            }
        ]
    })

    answer = result["messages"][-1].content
    return answer, metrics


**Invoke the transformed agent**

In [ ]:
answer, metrics = ask_agent_transformed(agents["Transformed"], "technology companies and business risks")
print_agent_result("Transformed", (answer, metrics))


**Invoke Agent (baseline, single-query retrieval)**

In [ ]:
import time

def timed_ask_agent(agent, question: str, pipeline_label: str):
    started = time.perf_counter()
    answer, metrics = ask_agent(agent, question)
    return {
        "pipeline": pipeline_label,
        "answer": answer,
        "metrics": metrics,
        "latency_seconds": round(time.perf_counter() - started, 3),
        "answer_tokens": count_tokens(answer),
    }

def timed_ask_agent_transformed(agent, question: str, pipeline_label: str):
    started = time.perf_counter()
    answer, metrics = ask_agent_transformed(agent, question)
    return {
        "pipeline": pipeline_label,
        "answer": answer,
        "metrics": metrics,
        "latency_seconds": round(time.perf_counter() - started, 3),
        "answer_tokens": count_tokens(answer),
    }

comparison_question = "technology companies and business risks"
baseline_result = timed_ask_agent(agents["Grounded"], comparison_question, "baseline (single-query)")
optimized_result = timed_ask_agent_transformed(agents["Transformed"], comparison_question, "transformed (baseline+rewrite+expansion+HyDE)")


**Print and compare results**

In [ ]:
comparison_df = pd.DataFrame(
    [
        {
            "pipeline": baseline_result["pipeline"],
            "latency_seconds": baseline_result["latency_seconds"],
            "answer_tokens": baseline_result["answer_tokens"],
        },
        {
            "pipeline": optimized_result["pipeline"],
            "latency_seconds": optimized_result["latency_seconds"],
            "answer_tokens": optimized_result["answer_tokens"],
        },
    ]
)

comparison_df


**Key takeaway**

Prompt engineering does not improve the underlying evidence. It improves how the agent retrieves, interprets, structures, and communicates that evidence. Strong prompts cannot compensate for poor retrieval, but they can make a well-retrieved answer more reliable, useful, and audience-appropriate.

Query transformation (rewriting, expansion, HyDE) attacks a different part of the problem: it improves the odds that the *right* evidence gets retrieved in the first place, before any prompt or context-optimization strategy gets a chance to work with it. Running all four retrieval techniques together and merging the results, as `search_news_archive_transformed` / `ask_agent_transformed` do, generally increases recall at the cost of several extra LLM calls (rewrite, expansion, HyDE) per question — a trade-off worth measuring explicitly, as the comparison cell above does, rather than assuming more techniques is always better.